In [1]:
import os
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
dataset_path = "/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/raw/UAH-DRIVESET-v1"

print(dataset_path)

/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/raw/UAH-DRIVESET-v1


In [3]:
drivers = sorted(

    d

    for d in os.listdir(dataset_path)

    if os.path.isdir(os.path.join(dataset_path, d))

    and d.startswith("D")

)

print(drivers)

['D1', 'D2', 'D3', 'D4', 'D5', 'D6']


In [4]:
trip_list = []

for driver in drivers:

    driver_path = os.path.join(
        dataset_path,
        driver
    )

    trips = sorted(

        trip

        for trip in os.listdir(driver_path)

        if os.path.isdir(
            os.path.join(driver_path, trip)
        )

    )

    for trip in trips:

        trip_list.append({

            "driver": driver,

            "trip": trip

        })

print(len(trip_list))

trip_list[:5]

40


[{'driver': 'D1', 'trip': '20151110175712-16km-D1-NORMAL1-SECONDARY'},
 {'driver': 'D1', 'trip': '20151110180824-16km-D1-NORMAL2-SECONDARY'},
 {'driver': 'D1', 'trip': '20151111123124-25km-D1-NORMAL-MOTORWAY'},
 {'driver': 'D1', 'trip': '20151111125233-24km-D1-AGGRESSIVE-MOTORWAY'},
 {'driver': 'D1', 'trip': '20151111132348-25km-D1-DROWSY-MOTORWAY'}]

In [21]:
from sklearn.model_selection import train_test_split

train_trips, test_trips = train_test_split(

    trip_list,

    test_size=0.20,

    random_state=42

)

print(len(train_trips))
print(len(test_trips))

32
8


In [5]:
def process_trip_v2(
    dataset_path,
    driver,
    trip
):

    trip_path = os.path.join(
        dataset_path,
        driver,
        trip
    )

    acc_path = os.path.join(
        trip_path,
        "RAW_ACCELEROMETERS.txt"
    )

    gps_path = os.path.join(
        trip_path,
        "RAW_GPS.txt"
    )

    # Load Sensors
    acc_df = load_accelerometer(acc_path)
    gps_df = load_gps(gps_path)

    # Synchronize
    master_df = synchronize_sensors(
        acc_df,
        gps_df
    )

    # Feature Engineering
    feature_df = engineer_features(master_df)

    # Advanced Sliding Window
    window_dataset = create_sliding_windows_v2(
        feature_df,
        window_features,
        WINDOW_SIZE
    )

    # Labels
    info = parse_trip_info(trip)

    window_dataset["driver"] = info["driver"]
    window_dataset["road_type"] = info["road_type"]
    window_dataset["behavior"] = simplify_behavior(
        info["behavior"]
    )

    return window_dataset

In [7]:
def load_gps(gps_path):

    gps_columns = [

        "timestamp",
        "speed",
        "latitude",
        "longitude",
        "altitude",

        "gps_quality",
        "satellites",

        "heading",

        "extra_1",
        "extra_2",
        "extra_3",
        "extra_4"

    ]

    gps_df = pd.read_csv(
        gps_path,
        sep=r"\s+",
        header=None,
        names=gps_columns
    )

    return gps_df

def synchronize_sensors(acc_df, gps_df):

    master_df = pd.merge_asof(

        acc_df.sort_values("timestamp"),

        gps_df.sort_values("timestamp"),

        on="timestamp",

        direction="nearest"

    )

    return master_df

def engineer_features(master_df):

    master_df = master_df.copy()

    # -------------------------------------------------
    # Acceleration Features
    # -------------------------------------------------

    master_df["acc_resultant"] = np.sqrt(
        master_df["acc_x"]**2 +
        master_df["acc_y"]**2 +
        master_df["acc_z"]**2
    )

    master_df["acc_horizontal"] = np.sqrt(
        master_df["acc_x"]**2 +
        master_df["acc_y"]**2
    )

    master_df["acc_vertical"] = master_df["acc_z"]

    # -------------------------------------------------
    # Delta Features
    # -------------------------------------------------

    master_df["speed_delta"] = master_df["speed"].diff().fillna(0)

    master_df["heading_delta"] = master_df["heading"].diff().fillna(0)

    master_df["roll_delta"] = master_df["roll"].diff().fillna(0)

    master_df["pitch_delta"] = master_df["pitch"].diff().fillna(0)

    master_df["yaw_delta"] = master_df["yaw"].diff().fillna(0)

    return master_df

def extract_statistics(signal):

    features = {}

    features["mean"] = signal.mean()

    features["std"] = signal.std()

    features["min"] = signal.min()

    features["max"] = signal.max()

    features["median"] = signal.median()

    features["rms"] = np.sqrt(
        np.mean(signal**2)
    )

    return features

def extract_window_features(window, feature_list):

    window_stats = {}

    for feature in feature_list:

        stats = extract_statistics(window[feature])

        for stat_name, stat_value in stats.items():

            column_name = f"{feature}_{stat_name}"

            window_stats[column_name] = stat_value

    return window_stats

def create_sliding_windows(
        feature_df,
        feature_list,
        window_size
):

    all_window_features = []

    for start in range(
        0,
        len(feature_df) - window_size + 1
    ):

        end = start + window_size

        window = feature_df.iloc[start:end]

        window_stats = extract_window_features(
            window,
            feature_list
        )

        all_window_features.append(window_stats)

    return pd.DataFrame(all_window_features)

def load_accelerometer(acc_path):

    acc_columns = [

        "timestamp",
        "active",

        "acc_x",
        "acc_y",
        "acc_z",

        "acc_x_kf",
        "acc_y_kf",
        "acc_z_kf",

        "roll",
        "pitch",
        "yaw"

    ]

    acc_df = pd.read_csv(
        acc_path,
        sep=r"\s+",
        header=None,
        names=acc_columns
    )

    return acc_df

In [10]:
def create_sliding_windows_v2(
    feature_df,
    feature_list,
    window_size
):

    all_window_features = []

    for start in range(
        0,
        len(feature_df) - window_size + 1
    ):

        end = start + window_size

        window = feature_df.iloc[start:end]

        window_stats = extract_window_features_v2(
            window,
            feature_list
        )

        all_window_features.append(
            window_stats
        )

    return pd.DataFrame(
        all_window_features
    )

In [12]:
def extract_statistics_v2(signal):

    features = {}

    # ------------------------------
    # Basic Statistics
    # ------------------------------

    features["mean"] = signal.mean()
    features["std"] = signal.std()
    features["variance"] = signal.var()

    features["min"] = signal.min()
    features["max"] = signal.max()

    features["median"] = signal.median()

    features["rms"] = np.sqrt(
        np.mean(signal ** 2)
    )

    # ------------------------------
    # Distribution Shape
    # ------------------------------

    features["skewness"] = signal.skew()

    features["kurtosis"] = signal.kurt()

    # ------------------------------
    # Percentiles
    # ------------------------------

    features["q25"] = signal.quantile(0.25)

    features["q75"] = signal.quantile(0.75)

    features["iqr"] = (
        features["q75"] -
        features["q25"]
    )

    return features

In [14]:
def extract_window_features_v2(window, feature_list):

    window_stats = {}

    for feature in feature_list:

        stats = extract_statistics_v2(window[feature])

        for stat_name, stat_value in stats.items():

            column_name = f"{feature}_{stat_name}"

            window_stats[column_name] = stat_value

    return window_stats

In [16]:
def parse_trip_info(trip_name):

    parts = trip_name.split("-")

    return {
        "date": parts[0],
        "distance": parts[1],
        "driver": parts[2],
        "behavior": parts[3],
        "road_type": parts[4]
    }

In [17]:
def simplify_behavior(label):

    if "NORMAL" in label:
        return "NORMAL"

    if "AGGRESSIVE" in label:
        return "AGGRESSIVE"

    if "DROWSY" in label:
        return "DROWSY"

    return label

In [18]:
def process_trip_v2(
    dataset_path,
    driver,
    trip
):

    trip_path = os.path.join(
        dataset_path,
        driver,
        trip
    )

    acc_path = os.path.join(
        trip_path,
        "RAW_ACCELEROMETERS.txt"
    )

    gps_path = os.path.join(
        trip_path,
        "RAW_GPS.txt"
    )

    # Load Sensors
    acc_df = load_accelerometer(acc_path)
    gps_df = load_gps(gps_path)

    # Synchronize
    master_df = synchronize_sensors(
        acc_df,
        gps_df
    )

    # Feature Engineering
    feature_df = engineer_features(master_df)

    # Advanced Sliding Window
    window_dataset = create_sliding_windows_v2(
        feature_df,
        window_features,
        WINDOW_SIZE
    )

    # Labels
    info = parse_trip_info(trip)

    window_dataset["driver"] = info["driver"]
    window_dataset["road_type"] = info["road_type"]
    window_dataset["behavior"] = simplify_behavior(
        info["behavior"]
    )

    return window_dataset

In [8]:
WINDOW_SIZE = 30

window_features = [
    "acc_resultant",
    "acc_horizontal",
    "speed",
    "speed_delta",
    "roll",
    "pitch",
    "yaw"
]

In [19]:
sample_driver = trip_list[0]["driver"]

sample_trip = trip_list[0]["trip"]

sample_dataset = process_trip_v2(
    dataset_path,
    sample_driver,
    sample_trip
)

print(sample_dataset.shape)

sample_dataset.head()

(6141, 87)


,acc_resultant_mean,acc_resultant_std,acc_resultant_variance,acc_resultant_min,acc_resultant_max,acc_resultant_median,acc_resultant_rms,acc_resultant_skewness,acc_resultant_kurtosis,acc_resultant_q25,...,yaw_median,yaw_rms,yaw_skewness,yaw_kurtosis,yaw_q25,yaw_q75,yaw_iqr,driver,road_type,behavior
0,0.041557,0.015206,0.000231,0.019339,0.075226,0.037947,0.044164,0.657782,-0.363049,0.029905,...,0.0215,0.019768,-0.472275,-1.474674,0.01325,0.024,0.01075,D1,SECONDARY,NORMAL
1,0.041732,0.015063,0.000227,0.019339,0.075226,0.037947,0.044282,0.663559,-0.315292,0.031588,...,0.0220,0.020169,-0.609582,-1.267201,0.01450,0.024,0.00950,D1,SECONDARY,NORMAL
2,0.041103,0.015089,0.000228,0.019339,0.075226,0.036665,0.043698,0.775687,-0.194939,0.031353,...,0.0220,0.020647,-0.699764,-0.982480,0.01625,0.024,0.00775,D1,SECONDARY,NORMAL
3,0.039805,0.015016,0.000225,0.019339,0.075226,0.034974,0.042455,0.904409,0.161935,0.029905,...,0.0220,0.021267,-0.612345,-0.581856,0.01725,0.024,0.00675,D1,SECONDARY,NORMAL
4,0.040324,0.015078,0.000227,0.019339,0.075226,0.036665,0.042963,0.794776,-0.023080,0.029905,...,0.0225,0.021915,-0.476406,-0.255128,0.01825,0.024,0.00575,D1,SECONDARY,NORMAL


In [22]:
train_datasets = []

for trip in train_trips:

    print(
        f"Processing Train: "
        f"{trip['driver']} - {trip['trip']}"
    )

    trip_dataset = process_trip_v2(
        dataset_path,
        trip["driver"],
        trip["trip"]
    )

    train_datasets.append(
        trip_dataset
    )

Processing Train: D6 - 20151221120051-26km-D6-AGGRESSIVE-MOTORWAY
Processing Train: D1 - 20151111135612-13km-D1-DROWSY-SECONDARY
Processing Train: D4 - 20151204152848-25km-D4-NORMAL-MOTORWAY
Processing Train: D2 - 20151120135152-25km-D2-DROWSY-MOTORWAY
Processing Train: D2 - 20151120164606-16km-D2-DROWSY-SECONDARY
Processing Train: D5 - 20151211162829-16km-D5-NORMAL1-SECONDARY
Processing Train: D5 - 20151211170502-16km-D5-DROWSY-SECONDARY
Processing Train: D2 - 20151120133502-26km-D2-AGGRESSIVE-MOTORWAY
Processing Train: D3 - 20151126125458-16km-D3-NORMAL2-SECONDARY
Processing Train: D4 - 20151203175637-17km-D4-DROWSY-SECONDARY
Processing Train: D1 - 20151110175712-16km-D1-NORMAL1-SECONDARY
Processing Train: D5 - 20151211165606-12km-D5-AGGRESSIVE-SECONDARY
Processing Train: D1 - 20151111134545-16km-D1-AGGRESSIVE-SECONDARY
Processing Train: D2 - 20151120162105-17km-D2-NORMAL2-SECONDARY
Processing Train: D1 - 20151110180824-16km-D1-NORMAL2-SECONDARY
Processing Train: D5 - 20151209153137-

In [23]:
train_dataset = pd.concat(
    train_datasets,
    ignore_index=True
)

print(train_dataset.shape)

train_dataset.head()

(243925, 87)


,acc_resultant_mean,acc_resultant_std,acc_resultant_variance,acc_resultant_min,acc_resultant_max,acc_resultant_median,acc_resultant_rms,acc_resultant_skewness,acc_resultant_kurtosis,acc_resultant_q25,...,yaw_median,yaw_rms,yaw_skewness,yaw_kurtosis,yaw_q25,yaw_q75,yaw_iqr,driver,road_type,behavior
0,0.058162,0.040807,0.001665,0.014213,0.178804,0.047605,0.070658,1.308464,1.423658,0.027403,...,-0.0335,0.033488,0.143241,-1.336769,-0.035,-0.03125,0.00375,D6,MOTORWAY,AGGRESSIVE
1,0.054311,0.033896,0.001149,0.014213,0.141287,0.047605,0.063721,0.984300,0.275005,0.027403,...,-0.0335,0.033431,0.292687,-1.044460,-0.035,-0.03125,0.00375,D6,MOTORWAY,AGGRESSIVE
2,0.055394,0.033481,0.001121,0.014213,0.141287,0.050094,0.064437,0.939426,0.288823,0.031732,...,-0.0335,0.033257,0.573495,-0.400818,-0.035,-0.03100,0.00400,D6,MOTORWAY,AGGRESSIVE
3,0.053637,0.033202,0.001102,0.014213,0.141287,0.047605,0.062790,1.093739,0.635064,0.031721,...,-0.0335,0.032998,0.832354,0.258202,-0.035,-0.03100,0.00400,D6,MOTORWAY,AGGRESSIVE
4,0.051045,0.031886,0.001017,0.014213,0.141287,0.044649,0.059903,1.305501,1.447782,0.029886,...,-0.0325,0.032622,1.001970,0.706236,-0.035,-0.03025,0.00475,D6,MOTORWAY,AGGRESSIVE


In [24]:
train_dataset["behavior"].value_counts()

,count
behavior,
NORMAL,116653
AGGRESSIVE,64031
DROWSY,63241


In [25]:
train_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 243925 entries, 0 to 243924
Data columns (total 87 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   acc_resultant_mean       243925 non-null  float64
 1   acc_resultant_std        243925 non-null  float64
 2   acc_resultant_variance   243925 non-null  float64
 3   acc_resultant_min        243925 non-null  float64
 4   acc_resultant_max        243925 non-null  float64
 5   acc_resultant_median     243925 non-null  float64
 6   acc_resultant_rms        243925 non-null  float64
 7   acc_resultant_skewness   243925 non-null  float64
 8   acc_resultant_kurtosis   243925 non-null  float64
 9   acc_resultant_q25        243925 non-null  float64
 10  acc_resultant_q75        243925 non-null  float64
 11  acc_resultant_iqr        243925 non-null  float64
 12  acc_horizontal_mean      243925 non-null  float64
 13  acc_horizontal_std       243925 non-null  float64
 14  acc_

In [26]:
test_datasets = []

for trip in test_trips:

    print(
        f"Processing Test: "
        f"{trip['driver']} - {trip['trip']}"
    )

    trip_dataset = process_trip_v2(
        dataset_path,
        trip["driver"],
        trip["trip"]
    )

    test_datasets.append(
        trip_dataset
    )

Processing Test: D3 - 20151126132013-17km-D3-DROWSY-SECONDARY
Processing Test: D3 - 20151126124208-16km-D3-NORMAL1-SECONDARY
Processing Test: D3 - 20151126113754-26km-D3-DROWSY-MOTORWAY
Processing Test: D4 - 20151204154908-25km-D4-AGGRESSIVE-MOTORWAY
Processing Test: D1 - 20151111132348-25km-D1-DROWSY-MOTORWAY
Processing Test: D2 - 20151120163350-16km-D2-AGGRESSIVE-SECONDARY
Processing Test: D6 - 20151221112434-17km-D6-NORMAL-SECONDARY
Processing Test: D4 - 20151204160823-25km-D4-DROWSY-MOTORWAY


In [27]:
test_dataset = pd.concat(
    test_datasets,
    ignore_index=True
)

print(test_dataset.shape)

test_dataset.head()

(66300, 87)


,acc_resultant_mean,acc_resultant_std,acc_resultant_variance,acc_resultant_min,acc_resultant_max,acc_resultant_median,acc_resultant_rms,acc_resultant_skewness,acc_resultant_kurtosis,acc_resultant_q25,...,yaw_median,yaw_rms,yaw_skewness,yaw_kurtosis,yaw_q25,yaw_q75,yaw_iqr,driver,road_type,behavior
0,0.090389,0.041935,0.001759,0.024779,0.146826,0.078529,0.099348,0.209095,-1.414184,0.056743,...,0.0195,0.020147,0.045145,-1.881546,0.01500,0.02400,0.00900,D3,SECONDARY,DROWSY
1,0.086940,0.041384,0.001713,0.024779,0.146826,0.073340,0.095990,0.338154,-1.307655,0.055493,...,0.0205,0.020517,-0.079191,-1.860689,0.01500,0.02400,0.00900,D3,SECONDARY,DROWSY
2,0.085278,0.041811,0.001748,0.024779,0.146826,0.070988,0.094669,0.430691,-1.299253,0.051903,...,0.0220,0.020924,-0.173227,-1.866020,0.01500,0.02475,0.00975,D3,SECONDARY,DROWSY
3,0.084539,0.040871,0.001670,0.024779,0.146826,0.070988,0.093603,0.426090,-1.253824,0.051903,...,0.0235,0.021322,-0.282224,-1.758559,0.01500,0.02500,0.01000,D3,SECONDARY,DROWSY
4,0.082301,0.039144,0.001532,0.024779,0.146826,0.070988,0.090855,0.520045,-1.020721,0.051903,...,0.0240,0.021755,-0.367356,-1.590613,0.01525,0.02500,0.00975,D3,SECONDARY,DROWSY


In [28]:
test_dataset["behavior"].value_counts()

,count
behavior,
DROWSY,36263
AGGRESSIVE,15317
NORMAL,14720


In [29]:
test_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66300 entries, 0 to 66299
Data columns (total 87 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   acc_resultant_mean       66300 non-null  float64
 1   acc_resultant_std        66300 non-null  float64
 2   acc_resultant_variance   66300 non-null  float64
 3   acc_resultant_min        66300 non-null  float64
 4   acc_resultant_max        66300 non-null  float64
 5   acc_resultant_median     66300 non-null  float64
 6   acc_resultant_rms        66300 non-null  float64
 7   acc_resultant_skewness   66300 non-null  float64
 8   acc_resultant_kurtosis   66300 non-null  float64
 9   acc_resultant_q25        66300 non-null  float64
 10  acc_resultant_q75        66300 non-null  float64
 11  acc_resultant_iqr        66300 non-null  float64
 12  acc_horizontal_mean      66300 non-null  float64
 13  acc_horizontal_std       66300 non-null  float64
 14  acc_horizontal_varianc

In [30]:
train_dataset.to_csv(
    "/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/processed/train_dataset_v2.csv",
    index=False
)

test_dataset.to_csv(
    "/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/processed/test_dataset_v2.csv",
    index=False
)

print("Datasets saved successfully!")

Datasets saved successfully!


In [31]:
print(train_dataset.shape)
print(test_dataset.shape)

print(train_dataset.columns.equals(test_dataset.columns))

(243925, 87)
(66300, 87)
True
